# Semantic Counting in Vector Databases — Demo

This notebook walks through the full pipeline:

1. **Load** the STS-B dataset
2. **Embed** sentences with `all-MiniLM-L6-v2`
3. **Cluster** via HDBSCAN
4. **Summarise** clusters with Google Colab's built-in LLM
5. **Query** — run `semantic_count(*)` for example queries
6. **Baseline** — compare against an embedding-only approach

No API keys required — uses `google.colab.ai.generate_text()` for all LLM calls.

## 0. Setup — Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/donggunkwak/Semantic-Count.git
%cd Semantic-Count
!pip install -q -r requirements.txt

In [ ]:
import sys, os

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

### Verify Colab AI is available

In [ ]:
from google.colab import ai

test = ai.generate_text("Say hello in one word.")
print(f"Colab AI test: {test}")

## 1. Load STS-B Sentences

In [ ]:
from src.data_loader import load_stsb_sentences

sentences = load_stsb_sentences()
print(f"Total unique sentences: {len(sentences)}")
print("\nFirst 5 sentences:")
for s in sentences[:5]:
    print(f"  \u2022 {s}")

## 2. Generate Embeddings

In [ ]:
from src.embeddings import generate_embeddings

embeddings = generate_embeddings(sentences)
print(f"Embeddings shape: {embeddings.shape}")

## 3. Cluster with HDBSCAN

In [ ]:
from src.clustering import cluster_embeddings, get_cluster_members

labels = cluster_embeddings(embeddings)

cluster_ids = sorted(set(labels) - {-1})
members = get_cluster_members(labels)

print(f"\nNumber of clusters: {len(cluster_ids)}")
print(f"Noise points: {labels.count(-1)}")
print(f"\nCluster sizes (top 10):")
sizes = sorted(((cid, len(members[cid])) for cid in cluster_ids), key=lambda x: -x[1])
for cid, size in sizes[:10]:
    print(f"  Cluster {cid}: {size} sentences")

## 4. Summarise Clusters (LLM)

This step uses Google Colab's built-in LLM to generate a short semantic summary for each cluster. Results are cached to `data/cluster_summaries.json`.

In [ ]:
from src.summarizer import summarize_clusters

summaries = summarize_clusters(sentences, labels)

print(f"\nCluster summaries ({len(summaries)} clusters):")
print("-" * 60)
for cid in sorted(summaries, key=int):
    print(f"  Cluster {cid}: {summaries[cid]}")
    print()

## 5. Semantic Counting — Example Queries

We run two example queries:
- **"sentences about happiness"**
- **"sentences about disagreement"**

For each query the engine:
1. Embeds the query
2. Finds the top-K nearest clusters
3. Filters clusters with the LLM
4. Checks individual documents in surviving clusters
5. Returns the semantic count

In [ ]:
from src.query_engine import semantic_count

queries = [
    "sentences about happiness",
    "sentences about disagreement",
]

results = {}
for q in queries:
    print("=" * 60)
    print(f'Query: "{q}"')
    print("=" * 60)
    result = semantic_count(
        q, sentences, embeddings, labels, summaries,
        save_path=None,
    )
    results[q] = result
    print()

### Display Results

In [ ]:
import pandas as pd

rows = []
for q, r in results.items():
    rows.append({
        "Query": q,
        "Clusters Retrieved": r.clusters_retrieved,
        "Clusters After LLM Filter": r.clusters_after_llm_filter,
        "Documents Checked": r.documents_checked,
        "semantic_count(*)": r.semantic_count,
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
for q, r in results.items():
    print(f'\n--- Cluster details for "{q}" ---')
    for c in r.cluster_details:
        status = "KEPT" if c["relevant"] else "FILTERED"
        print(f"  [{status}] Cluster {c['cluster_id']}: {c['summary']}")

    print(f"\n  Matched sentences ({r.semantic_count}):")
    for s in r.matched_sentences[:10]:
        print(f"    \u2022 {s}")
    if r.semantic_count > 10:
        print(f"    ... and {r.semantic_count - 10} more")

## 6. Baseline Comparison (No LLM)

The baseline counts documents whose embedding cosine similarity to the query exceeds a fixed threshold. No LLM calls are made.

In [ ]:
from src.baseline import baseline_count

baseline_results = {}
for q in queries:
    print(f'Baseline for "{q}":')
    br = baseline_count(q, sentences, embeddings, threshold=0.45)
    baseline_results[q] = br
    print()

In [ ]:
comparison_rows = []
for q in queries:
    comparison_rows.append({
        "Query": q,
        "LLM semantic_count": results[q].semantic_count,
        "Baseline count (cos>0.45)": baseline_results[q].semantic_count,
        "LLM docs checked": results[q].documents_checked,
    })

df_cmp = pd.DataFrame(comparison_rows)
display(df_cmp)

## Summary

The LLM-based semantic counting pipeline:
- Uses clustering to **prune the search space** (only check relevant clusters)
- Uses LLM calls at two levels: **cluster filtering** (cheap) and **document checking** (precise)
- Produces more semantically accurate counts than a simple cosine threshold

The baseline is fast and requires no LLM calls, but cannot capture nuanced semantic distinctions.